# EPI Vision — retreino 2: luvas e botas

Continua do modelo **atual** (`best_cores.pt`, o que reconhece colete azul), sem treinar do zero, e acrescenta **Luvas** e **Botas**.

**Antes de rodar:** Ambiente de execução → Alterar tipo → **GPU (T4)**.

**Como funciona (resumo):**
1. Baixa o dataset público *Construction-PPE* (Ultralytics, CC BY 4.0: tem `gloves`, `boots`, `no_gloves`, `no_boots`).
2. Treina um detector **só** de luvas/botas nele (etapa A).
3. Usa esse detector para **pré-rotular luvas e botas nas imagens antigas** (só dentro do corpo de uma pessoa, confiança ≥ 0,60). Sem isso, as imagens antigas ensinariam "luva sem rótulo = fundo".
4. Treina o modelo final com tudo junto e **compara com o resultado atual** (Hardhat 0,889 / Safety Vest 0,918 / Person 0,883). Se piorar, NÃO troque o modelo.

**Limitações (leia):**
- O Construction-PPE tem só 1.000 imagens e parte delas é **gerada por IA**, com enquadramento próximo. Foto real do porto vale mais.
- "Botas" aqui é bota de trabalho em geral. Bota de segurança certificada (biqueira, solado) só se confirma com o porto.
- Câmera de portaria costuma NÃO ver mãos/pés: nesses casos o sistema marca "não observável", nunca "ausente".

In [ ]:
!nvidia-smi | head -15
!pip -q install ultralytics roboflow

## 1. Enviar o modelo atual
Envie o `best_cores.pt` (o que você baixou na rodada 1).

In [ ]:
from google.colab import files
up = files.upload()
BEST = list(up.keys())[0]
print('modelo base:', BEST)

## 2. Dataset base (o misturado da rodada 1)
- Se o Colab da rodada 1 **ainda está conectado**, a pasta `/content/merged` existe e esta etapa é pulada.
- Se não existe, ele é **reconstruído** (precisa da chave do Roboflow — use a **NOVA**, a antiga deve ser revogada).

In [ ]:
import os
HAS_MERGED = os.path.isdir('/content/merged/train/images')
print('dataset base já existe:', HAS_MERGED)

In [ ]:
# Só roda se HAS_MERGED for False. Reconstrói o dataset da rodada 1.
import shutil, yaml, glob
from roboflow import Roboflow
from ultralytics import YOLO
if not HAS_MERGED:
    API_KEY = 'COLE_SUA_CHAVE_NOVA_AQUI'
    rf = Roboflow(api_key=API_KEY)
    old = rf.workspace('roboflow-universe-projects').project('construction-site-safety').version(27).download('yolov8', location='/content/old')
    newd = rf.workspace('college-2t4kc').project('safety-vest-detection').version(1).download('yolov8', location='/content/new0')
    def _names(p):
        n = yaml.safe_load(open(p + '/data.yaml'))['names']
        return list(n.values()) if isinstance(n, dict) else list(n)
    _base = _names(old.location); _low = [n.lower() for n in _base]
    _cid = lambda n: _low.index(n.lower())
    det0 = YOLO(BEST); OUTM = '/content/merged'
    for s in ('train','valid','test'):
        for k in ('images','labels'): os.makedirs(f'{OUTM}/{s}/{k}', exist_ok=True)
    for s in ('train','valid','test'):
        for k in ('images','labels'):
            src = f'{old.location}/{s}/{k}'
            if os.path.isdir(src):
                for f in os.listdir(src): shutil.copy(f'{src}/{f}', f'{OUTM}/{s}/{k}/old_{f}')
    pid = _cid('Person')
    for s in ('train','valid','test'):
        si, sl = f'{newd.location}/{s}/images', f'{newd.location}/{s}/labels'
        if not os.path.isdir(si): continue
        for f in os.listdir(si):
            stem = os.path.splitext(f)[0]; lines = []
            lp = f'{sl}/{stem}.txt'
            if os.path.exists(lp):
                for l in open(lp).read().splitlines():
                    r = l.split()
                    if len(r) >= 5 and int(float(r[0])) in (0, 1):
                        lines.append(' '.join([str(_cid({0:'Hardhat',1:'Safety Vest'}[int(float(r[0]))]))] + r[1:5]))
            res = det0.predict(f'{si}/{f}', conf=0.60, classes=[pid], verbose=False)[0]
            for b in res.boxes.xywhn.tolist(): lines.append(f'{pid} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}')
            shutil.copy(f'{si}/{f}', f'{OUTM}/{s}/images/new0_{f}')
            open(f'{OUTM}/{s}/labels/new0_{stem}.txt', 'w').write('\n'.join(lines))
    yaml.safe_dump({'path': OUTM, 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
                    'names': {i: n for i, n in enumerate(_base)}}, open(f'{OUTM}/data.yaml', 'w'))
    print('dataset base reconstruído')

In [ ]:
import yaml
BASE_NAMES = yaml.safe_load(open('/content/merged/data.yaml'))['names']
BASE_NAMES = list(BASE_NAMES.values()) if isinstance(BASE_NAMES, dict) else list(BASE_NAMES)
print(BASE_NAMES)
assert BASE_NAMES[:10] == ['Hardhat','Mask','NO-Hardhat','NO-Mask','NO-Safety Vest','Person','Safety Cone','Safety Vest','machinery','vehicle'], 'ordem de classes inesperada'
n_img = {s: len(os.listdir(f'/content/merged/{s}/images')) for s in ('train','valid','test')}
print(n_img)

## 3. Baixar o Construction-PPE (Ultralytics, CC BY 4.0)
Atribuição obrigatória: *Ultralytics Construction-PPE Dataset, Ultralytics team, CC BY 4.0.* Mantenha este crédito na documentação do projeto.

In [ ]:
!wget -q -O /content/cppe.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip
!unzip -q -o /content/cppe.zip -d /content/cppe
!ls /content/cppe/construction-ppe

## 4. Funções auxiliares (testadas antes de entrar aqui)

In [ ]:
# Funções puras (testadas no sandbox antes de ir para o notebook)
import os, glob, shutil

def read_labels(path):
    out = []
    if os.path.exists(path):
        for l in open(path).read().splitlines():
            r = l.split()
            if len(r) >= 5:
                out.append((int(float(r[0])), *[float(v) for v in r[1:5]]))
    return out

def center_inside_any(box, persons, margin=0.0):
    """box = (cx, cy, w, h) normalizado; True se o CENTRO cai dentro de alguma pessoa."""
    cx, cy = box[0], box[1]
    for (_, px, py, pw, ph) in persons:
        if abs(cx - px) <= pw / 2 + margin and abs(cy - py) <= ph / 2 + margin:
            return True
    return False

def convert_cppe_label(src_txt, cmap, names_out):
    """Converte um .txt do Construction-PPE para os ids do dataset final. Classes fora do mapa são descartadas."""
    lines = []
    for (c, cx, cy, w, h) in read_labels(src_txt):
        if c in cmap:
            lines.append(f"{names_out.index(cmap[c])} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines

## 5. Etapa A — detector de luvas e botas (só Construction-PPE)
Parte do `best_cores.pt`. Serve apenas para **pré-rotular** as imagens antigas na etapa seguinte.

In [ ]:
import yaml
from ultralytics import YOLO
CP = '/content/cppe/construction-ppe'
yaml.safe_dump({'path': CP, 'train': 'images/train', 'val': 'images/val',
                'names': {0:'helmet',1:'gloves',2:'vest',3:'boots',4:'goggles',5:'Person',6:'no_helmet',
                          7:'no_goggle',8:'no_gloves',9:'no_boots',10:'forklift',11:'no_vest'}},
               open('/content/cppe_abs.yaml', 'w'))
ma = YOLO(BEST)
ra = ma.train(data='/content/cppe_abs.yaml', epochs=40, imgsz=640, batch=16, patience=12,
              project='/content/runs', name='gb_detector', device=0)

In [ ]:
gb = YOLO('/content/runs/gb_detector/weights/best.pt')
m = gb.val(data='/content/cppe_abs.yaml')
for i, n in enumerate(m.names.values()):
    if n in ('gloves', 'boots', 'no_gloves', 'no_boots'):
        print(f'{n:10s} mAP50={m.box.ap50[i]:.3f}')

## 6. Etapa B — pré-rotular luvas e botas nas imagens antigas
Só entram caixas com confiança ≥ `GB_CONF` **cujo centro está dentro de uma pessoa já rotulada** (corta falsos positivos soltos pelo cenário).
Aumente `GB_CONF` (ex.: 0,70) se a amostra visual mostrar muito erro.

In [ ]:
import glob, shutil
GB_CONF = 0.60
NAMES = BASE_NAMES + ['Gloves', 'Boots', 'NO-Gloves', 'NO-Boots']
GLOVES, BOOTS = NAMES.index('Gloves'), NAMES.index('Boots')
PERSON = NAMES.index('Person')
gb_ids = {i: n for i, n in gb.names.items() if n in ('gloves', 'boots')}
gb_class_ids = list(gb_ids.keys())

FINAL = '/content/final'
for s in ('train', 'valid', 'test'):
    for k in ('images', 'labels'):
        os.makedirs(f'{FINAL}/{s}/{k}', exist_ok=True)

added = {'Gloves': 0, 'Boots': 0}
for s in ('train', 'valid', 'test'):
    for im in glob.glob(f'/content/merged/{s}/images/*'):
        stem = os.path.splitext(os.path.basename(im))[0]
        lab = f'/content/merged/{s}/labels/{stem}.txt'
        lines = open(lab).read().splitlines() if os.path.exists(lab) else []
        persons = [l for l in read_labels(lab) if l[0] == PERSON]
        if persons:
            r = gb.predict(im, conf=GB_CONF, classes=gb_class_ids, verbose=False)[0]
            for c, b in zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist()):
                if center_inside_any(b, persons):
                    nm = 'Gloves' if gb_ids[int(c)] == 'gloves' else 'Boots'
                    lines.append(f'{NAMES.index(nm)} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}')
                    added[nm] += 1
        shutil.copy(im, f'{FINAL}/{s}/images/{os.path.basename(im)}')
        open(f'{FINAL}/{s}/labels/{stem}.txt', 'w').write('\n'.join(lines))
print('caixas pré-rotuladas nas imagens antigas:', added)

## 7. Juntar o Construction-PPE (rótulos reais dele) ao dataset final
- `train` do Construction-PPE entra **2 vezes** (é pequeno e é a única fonte de luvas/botas rotuladas à mão).
- `val` do Construction-PPE entra em `valid`.
- Classes dele que não usamos (`goggles`, `no_goggle`, `forklift`) são descartadas.

In [ ]:
CMAP = {0:'Hardhat', 1:'Gloves', 2:'Safety Vest', 3:'Boots', 5:'Person',
        6:'NO-Hardhat', 8:'NO-Gloves', 9:'NO-Boots', 11:'NO-Safety Vest'}
for split_src, split_dst, reps in (('train', 'train', ('a', 'b')), ('val', 'valid', ('a',))):
    for f in os.listdir(f'{CP}/images/{split_src}'):
        stem = os.path.splitext(f)[0]
        lines = convert_cppe_label(f'{CP}/labels/{split_src}/{stem}.txt', CMAP, NAMES)
        for rep in reps:
            shutil.copy(f'{CP}/images/{split_src}/{f}', f'{FINAL}/{split_dst}/images/cppe{rep}_{f}')
            open(f'{FINAL}/{split_dst}/labels/cppe{rep}_{stem}.txt', 'w').write('\n'.join(lines))

yaml.safe_dump({'path': FINAL, 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
                'names': {i: n for i, n in enumerate(NAMES)}}, open(f'{FINAL}/data.yaml', 'w'))
print({s: len(os.listdir(f'{FINAL}/{s}/images')) for s in ('train', 'valid', 'test')})
print(NAMES)

### 7b. Conferir uma amostra (imagens ANTIGAS com luvas/botas pré-rotuladas)
Se muitas caixas de luva/bota estiverem erradas, aumente `GB_CONF` na etapa 6 e rode de novo a partir dela.

In [ ]:
import random, cv2
from IPython.display import display
from PIL import Image
cand = [p for p in glob.glob(f'{FINAL}/train/images/old_*')
        if any(l.startswith((str(GLOVES), str(BOOTS))) for l in open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt').read().splitlines())]
random.seed(2)
print(len(cand), 'imagens antigas com luva/bota pré-rotulada')
for p in random.sample(cand, min(8, len(cand))):
    im = cv2.imread(p); h, w = im.shape[:2]
    for l in open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt').read().splitlines():
        c, x, y, bw, bh = l.split(); c = int(c); x, y, bw, bh = [float(v) for v in (x, y, bw, bh)]
        if c in (GLOVES, BOOTS):
            p1, p2 = (int((x-bw/2)*w), int((y-bh/2)*h)), (int((x+bw/2)*w), int((y+bh/2)*h))
            cv2.rectangle(im, p1, p2, (0, 255, 0), 3); cv2.putText(im, NAMES[c], p1, 0, 0.8, (0, 255, 0), 2)
    display(Image.fromarray(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)).resize((560, int(560*h/w))))

## 8. Treinar o modelo final
Parte do `best_cores.pt`. Mesmos ajustes de cor da rodada 1 (funcionaram para o colete azul).

In [ ]:
model = YOLO(BEST)
res = model.train(data=f'{FINAL}/data.yaml', epochs=50, imgsz=640, batch=16, patience=15,
                  hsv_h=0.5, hsv_s=0.7, hsv_v=0.4, project='/content/runs', name='finetune_gb', device=0)

## 9. Avaliar e comparar com o modelo atual
**Regra de decisão:** se Hardhat, Safety Vest ou Person caírem mais de 0,03 em relação ao atual, **não troque o modelo**.
Obs.: o `valid` agora inclui imagens do Construction-PPE (parte gerada por IA) e rótulos pré-feitos, então os números não são idênticos aos da rodada 1. O teste que decide é o da câmera.

In [ ]:
new = YOLO('/content/runs/finetune_gb/weights/best.pt')
m = new.val(data=f'{FINAL}/data.yaml')
AP = {n: float(m.box.ap50[i]) for i, n in enumerate(m.names.values())}
for n, v in AP.items(): print(f'{n:15s} mAP50={v:.3f}')
print('mAP50 geral:', round(float(m.box.map50), 3))
REF = {'Hardhat': 0.889, 'Safety Vest': 0.918, 'Person': 0.883}
print()
ok = True
for n, ref in REF.items():
    d = AP[n] - ref
    flag = 'OK' if d >= -0.03 else 'PIOROU — NÃO TROQUE'
    ok &= d >= -0.03
    print(f'{n:12s} antes {ref:.3f}  agora {AP[n]:.3f}  ({d:+.3f})  {flag}')
print()
print('RESULTADO:', 'pode testar na câmera' if ok else 'NÃO use este modelo (regressão nas classes principais)')

## 10. Testar nas SUAS imagens (o que decide)
Envie fotos **reais**, que não entraram no treino:
- mão com luva · mão sem luva · bota de trabalho · tênis comum · pessoa de corpo inteiro
- **regressão:** colete azul, colete laranja, capacete (o modelo não pode ter piorado nisso)
O que olhar: `Gloves`/`Boots` só no que é luva/bota; tênis comum e mão nua NÃO devem virar luva/bota com confiança alta.

In [ ]:
up = files.upload()
for name in up:
    r = new.predict(name, conf=0.25, save=True, project='/content/pred', name='teste2', exist_ok=True)
    print(name, [(new.names[int(c)], round(float(s), 2)) for c, s in zip(r[0].boxes.cls, r[0].boxes.conf)])
from IPython.display import Image as IImage, display
for f in sorted(glob.glob('/content/pred/teste2/*')):
    display(IImage(f, width=480))

## 11. Baixar o novo modelo
Salve como `best_cores_gb.pt` (não sobrescreva o `best_cores.pt` que está funcionando).

In [ ]:
files.download('/content/runs/finetune_gb/weights/best.pt')